<a href="https://colab.research.google.com/github/23xavimaya/aiatemasejercicio/blob/main/2026_1_Ejercicios_Scripting_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ejercicio 1: “Auditor de logs” (find + grep + date + redirección + if + función)**

---


**Objetivo:** Buscar patrones en archivos `.log` dentro de una carpeta y generar un reporte fechado.

**Requisitos:**

* Variables: `RUTA`, `PATRON`, `SALIDA`
* if para validar que la ruta existe y que el patrón no esté vacío.
* Función `generar_reporte()` que:
  * use `find` para localizar `*.log`
  * use `grep -n` para buscar `PATRON`
  * guarde resultados en un archivo `reporte_YYYY-MM-DD.txt` usando date

**Redirección:**

* stdout a reporte (>)
* errores de `find` a `evidencia/errores_find.txt` (2>)

**Extensión (opcional):**
* Si no hay coincidencias, escribir “SIN HALLAZGOS” en el reporte.


In [ ]:
#!/bin/bash
RUTA="logs"
PATRON="ERROR"
SALIDA="reporte_$(date +%F).txt"
mkdir -p evidencia
if [ ! -d "$RUTA" ]; then
    echo "La ruta no existe."
    exit 1
fi
if [ -z "$PATRON" ]; then
    echo "El patrón está vacío."
    exit 1
fi
generar_reporte() {
    # Buscar todos los archivos .log y buscar el patrón
    find "$RUTA" -name "*.log" 2> evidencia/errores_find.txt | while read -r archivo
    do
        grep -n "$PATRON" "$archivo"
    done > "$SALIDA"

    # Si no hubo coincidencias
    if [ ! -s "$SALIDA" ]; then
        echo "SIN HALLAZGOS" > "$SALIDA"
    fi
    echo "Reporte generado: $SALIDA"
}
generar_reporte

# **Ejercicio 2: “Normalizador de textos” (sed + for + if + redirección)**

---

**Objetivo:** Normalizar archivos `.txt` en una carpeta: quitar espacios duplicados y convertir a minúsculas (o reemplazar una palabra por otra).

**Requisitos:**

* Variables: `IN_DIR`, `OUT_DIR`
* `if` para crear `OUT_DIR` si no existe.
* `for` para iterar archivos `*.txt` encontrados con find.
* `sed` para:
  * reemplazar múltiples espacios por uno (por ejemplo, `s/[[:space:]]\+/ /g`)
  * (opcional) reemplazar “ERROR” por “ALERTA”

* Redirección: guardar cada archivo procesado en `OUT_DIR/nombre_normalizado.txt`

**Entrega:**
* `OUT_DIR/` con los archivos resultantes.


In [ ]:
#!/bin/bash
IN_DIR="textos"
OUT_DIR="textos_normalizados"
if [ ! -d "$OUT_DIR" ]; then
    mkdir "$OUT_DIR"
fi
for archivo in $(find "$IN_DIR" -name "*.txt")
do
    nombre=$(basename "$archivo")
    sed 's/[[:space:]]\+/ /g; s/ERROR/ALERTA/g' "$archivo" | tr '[:upper:]' '[:lower:]' > "$OUT_DIR/$nombre"
    echo "Archivo procesado: $nombre"
done
echo "Proceso finalizado."

# **Ejercicio 3: “Menú interactivo de mantenimiento” (while + case/if + funciones + export)**
---

**Objetivo:** Crear un menú en consola que permita ejecutar acciones repetidas hasta que el usuario salga.

**Requisitos:**

* `while true` para mantener el menú.

* Funciones:

  * `mostrar_fecha()` (usa date)

  * `buscar_archivo()` (usa find)

  * `buscar_texto()` (usa grep)

* Variable exportada: `export HIST_DIR="$HOME/so_lab_hist"`

* Redirección: registrar cada acción en `evidencia/acciones.log` con >> (incluye timestamp con date).

**Opciones del menú:**

1. Mostrar fecha/hora
2. Buscar archivo por nombre en una ruta dada
3. Buscar texto en un archivo dado
4. Salir


In [ ]:
#!/bin/bash
export HIST_DIR="$HOME/so_lab_hist"
mkdir -p evidencia
mostrar_fecha() {
    date
    echo "$(date) - Se mostró la fecha" >> evidencia/acciones.log
}
buscar_archivo() {
    echo "Ingrese la ruta:"
    read ruta
    echo "Ingrese el nombre del archivo:"
    read nombre
    find "$ruta" -name "$nombre"
    echo "$(date) - Búsqueda de archivo: $nombre" >> evidencia/acciones.log
}

buscar_texto() {
    echo "Ingrese el archivo:"
    read archivo
    echo "Ingrese el texto a buscar:"
    read texto
    grep -n "$texto" "$archivo"
    echo "$(date) - Búsqueda del texto: $texto" >> evidencia/acciones.log
}

while true
do
    echo "==========================="
    echo " MENÚ DE MANTENIMIENTO"
    echo "==========================="
    echo "1. Mostrar fecha y hora"
    echo "2. Buscar archivo"
    echo "3. Buscar texto"
    echo "4. Salir"
    echo "Seleccione una opción:"
    read opcion
    case $opcion in
        1)
            mostrar_fecha
            ;;
        2)
            buscar_archivo
            ;;
        3)
            buscar_texto
            ;;
        4)
            echo "Saliendo..."
            break
            ;;
        *)
            echo "Opción no válida."
            ;;
    esac

    echo
done

# **Ejercicio 4: “Contador de coincidencias por archivo” (find + grep + for + variables + redirección)**
---

**Objetivo:** Para cada archivo `*.conf` o `*.txt` en una ruta, contar cuántas líneas contienen un patrón.

**Requisitos:**
* Variables: `RUTA`, `PATRON`
* for sobre resultados de find
* Usar `grep -c` (o `grep | wc -l` si aún no ven `wc`)
* Guardar una tabla en `evidencia/conteo.txt` con formato:
  * archivo | coincidencias
* if para omitir archivos vacíos o no legibles.

In [ ]:
#!/bin/bash
RUTA="."
PATRON="ERROR"
mkdir -p evidencia
echo "Archivo | Coincidencias" > evidencia/conteo.txt
for archivo in $(find "$RUTA" \( -name "*.conf" -o -name "*.txt" \))
do
    if [ -r "$archivo" ] && [ -s "$archivo" ]; then
        cantidad=$(grep -c "$PATRON" "$archivo")
        echo "$archivo | $cantidad" >> evidencia/conteo.txt

    fi
done
echo "Conteo terminado."

# **Ejercicio 5: “Renombrador con fecha y saneamiento” (date + sed + while + if)**
---

**Objetivo:** Renombrar archivos de una carpeta agregando prefijo de fecha y eliminando caracteres “problemáticos” del nombre.

**Requisitos:**
* Variables: `DIR`, `PREFIJO="$(date +%Y%m%d)"`
* `find` para listar archivos (no directorios).
* `while` `read -r archivo; do ... done` para procesar rutas (en vez de `for`).
* `sed` para transformar el nombre:
  * espacios a guiones
  * eliminar dobles guiones

* `if` para evitar colisiones: si el nuevo nombre ya existe, agregar sufijo incremental.

**Salida:**
* `evidencia/renombrados.txt` con `viejo -> nuevo`



In [ ]:
#!/bin/bash
DIR="archivos"
PREFIJO="$(date +%Y%m%d)"
mkdir -p evidencia
> evidencia/renombrados.txt
find "$DIR" -type f | while read -r archivo
do
    nombre=$(basename "$archivo")
    ruta=$(dirname "$archivo")
    nuevo_nombre=$(echo "$nombre" | sed 's/ /-/g; s/--/-/g')
    nuevo_nombre="${PREFIJO}_${nuevo_nombre}"
    destino="$ruta/$nuevo_nombre"
    contador=1
    while [ -e "$destino" ]
    do
        destino="$ruta/${PREFIJO}_${contador}_$nuevo_nombre"
        contador=$((contador + 1))
    done
    mv "$archivo" "$destino"
    echo "$archivo -> $destino" >> evidencia/renombrados.txt

done

echo "Renombrado finalizado."

# **Ejercicio 6: “Extractor de configuración” (grep + sed + funciones + redirección)**
---

**Objetivo:** Dado un archivo de configuración estilo `CLAVE=VALOR`, producir un `.env` limpio y exportable.

**Requisitos:**
* Función `limpiar_env()` que:

  * use `grep -v` para excluir líneas vacías y comentarios (^#)

  * use `sed` para recortar espacios alrededor de =

* Crear `evidencia/app.env`

* Luego, `export` de variables cargándolas desde `app.env` (por ejemplo, con `set -a`; `source evidencia/app.env; set +a`)

* `if` para validar que el archivo existe.

**Comprobación:**
* `echo` de 2 variables exportadas para verificar.

In [ ]:
#!/bin/bash
ARCHIVO="config.txt"
mkdir -p evidencia
if [ ! -f "$ARCHIVO" ]; then
    echo "El archivo no existe."
    exit 1
fi
limpiar_env() {
    grep -v "^#" "$ARCHIVO" | \
    grep -v "^$" | \
    sed 's/[[:space:]]*=[[:space:]]*/=/g' > evidencia/app.env
}
limpiar_env
set -a
source evidencia/app.env
set +a
echo "Variable 1: $USUARIO"
echo "Variable 2: $PUERTO"

# **Ejercicio 7: “Buscador de patrones con rotación de reportes” (date + if + while + redirecciones)**
---

**Objetivo:** Generar reportes diarios; si el reporte del día existe, rotarlo a .bak y crear uno nuevo.

**Requisitos:**
* Variables: `RUTA`, `PATRON`, `REPORTE="evidencia/reporte_$(date +%F).txt`"

* `if`:

  * si existe `REPORTE`, mover a `REPORTE.bak`

* `while read -r file; do` ... leyendo salida de `find`

* `grep -n` y `echo` para armar un reporte por secciones:

  * encabezado con timestamp

  * por cada archivo: nombre y coincidencias

* Redirección combinada sugerida: >> y 2>> a `evidencia/errores.txt`

In [ ]:
#!/bin/bash
RUTA="logs"
PATRON="ERROR"
REPORTE="evidencia/reporte_$(date +%F).txt"
mkdir -p evidencia
if [ -f "$REPORTE" ]; then
    mv "$REPORTE" "$REPORTE.bak"
fi
echo "==============================" > "$REPORTE"
echo "REPORTE DE BÚSQUEDA" >> "$REPORTE"
echo "Fecha: $(date)" >> "$REPORTE"
echo "==============================" >> "$REPORTE"
echo "" >> "$REPORTE"
find "$RUTA" -name "*.log" 2>> evidencia/errores.txt | while read -r archivo
do
    echo "Archivo: $archivo" >> "$REPORTE"
    grep -n "$PATRON" "$archivo" >> "$REPORTE" 2>> evidencia/errores.txt
    echo "" >> "$REPORTE"
done
echo "Reporte generado correctamente."

# **Ejercicio 8: “Mini-ETL de texto” (find + grep + sed + for + funciones + export)**
---

**Objetivo:** Recorrer archivos `.txt`, filtrar líneas que contienen un patrón, aplicar una transformación y consolidar todo en un solo archivo.

**Requisitos:**
* `export ETL_OUT="evidencia/consolidado.txt"`

* Función `transformar_linea()` que use `sed` (por ejemplo, reemplazar tabs por espacios, o anonimizar emails simple).

* Pipeline por archivo:

  * `grep` filtra

  * `sed` transforma

  * redirección `>>` agrega al consolidado

* `if` para crear/limpiar el consolidado al inicio (`> "$ETL_OUT"`)

In [1]:
#!/bin/bash
export ETL_OUT="evidencia/consolidado.txt"
RUTA="textos"
PATRON="ERROR"
mkdir -p evidencia
> "$ETL_OUT"
transformar_linea() {
    sed 's/\t/ /g'
}
for archivo in $(find "$RUTA" -name "*.txt")
do
    echo "===== $archivo =====" >> "$ETL_OUT"
    grep "$PATRON" "$archivo" | transformar_linea >> "$ETL_OUT"
    echo "" >> "$ETL_OUT"
done
echo "Proceso ETL finalizado."

SyntaxError: invalid syntax (3941244861.py, line 2)

#**PLUS: Script para aleatorizar una lista de nombres**

Selecciona al azar de una lista de nombres en un archivo.
No debe repetirse los nombres que van saliendo, debe salir una única vez.
Además, almacenar a parte los nombres en el orden que van saliendo.

In [ ]:
#!/bin/bash
ARCHIVO="nombres.txt"
SALIDA="orden_salida.txt"
if [ ! -f "$ARCHIVO" ]; then
    echo "El archivo no existe."
    exit 1
fi
> "$SALIDA"
while read -r nombre
do
    echo "$nombre"
    echo "$nombre" >> "$SALIDA"
done < <(shuf "$ARCHIVO")

echo
echo "Todos los nombres fueron seleccionados."
echo "El orden quedó guardado en $SALIDA."